# Training Shelf Detection (Colab atau lokal)

Notebook ini menangani training detector YOLOv8m Shelf Detection dengan konfigurasi yang diselaraskan dengan repo referensi planogram compliance. Hasil akhirnya adalah `best.pt` yang disimpan permanen di storage persisten: Google Drive saat di Colab, atau folder project saat di luar Colab. Export ONNX, FAISS, API, dan deployment tetap tidak dijalankan di notebook ini.

Jalankan setiap cell berurutan dari atas ke bawah. Untuk smoke test cepat, ubah `EPOCHS` menjadi 5 sementara; konfigurasi default mengikuti training penuh 100 epoch.

## 1. Pilih GPU runtime

Jika berjalan di Google Colab, buka **Runtime > Change runtime type**, lalu pilih **T4 GPU** atau GPU lain yang tersedia. Jika berjalan lokal, pastikan environment Python sudah memakai PyTorch CUDA dan driver NVIDIA yang cocok. Training YOLOv8m dengan CPU akan sangat lambat.

## 2. Konfigurasi utama

Cell berikut mengumpulkan seluruh nilai yang mungkin perlu diubah:

- `MODEL_NAME`: checkpoint YOLO yang dipakai. Default tetap `yolov8m.pt` agar sejalan dengan konfigurasi referensi.
- `DATASET_NAME`: descriptor resmi SKU-110K dari Ultralytics yang akan diunduh otomatis.
- `SAVE_SKU110K_TO_DRIVE`: simpan dataset yang sudah disiapkan ke root persisten. Di Colab ini Google Drive; di luar Colab ini folder project lokal.
- `RESTORE_SKU110K_FROM_DRIVE`: salin cache dataset ke runtime lokal sebelum training agar akses file tetap cepat.
- `EPOCHS`, `BATCH`, `LR0`, `LRF`, `MOMENTUM`, `WEIGHT_DECAY`, `WARMUP_EPOCHS`, dan `PATIENCE`: hyperparameter yang mengikuti konfigurasi referensi.
- `USE_WANDB`: aktifkan dashboard online W&B jika API key tersedia dari Colab Secrets atau env var lokal.
- `RESUME_CHECKPOINT`: isi dengan path `last.pt` bila ingin melanjutkan training yang terputus.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone

def detect_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


IS_COLAB = detect_colab()
LOCAL_PROJECT_ROOT = Path.cwd().resolve()
for candidate in (LOCAL_PROJECT_ROOT, *LOCAL_PROJECT_ROOT.parents):
    if (candidate / ".git").is_dir() and (candidate / "notebooks").is_dir():
        LOCAL_PROJECT_ROOT = candidate
        break
COLAB_RUNTIME_ROOT = Path("/content")
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/shelf-detection")
RUNTIME_ROOT = COLAB_RUNTIME_ROOT if IS_COLAB else LOCAL_PROJECT_ROOT
DRIVE_ROOT = COLAB_DRIVE_ROOT if IS_COLAB else LOCAL_PROJECT_ROOT

REPO_URL = "https://github.com/kholiqdev/shelf-detection.git"
REPO_DIR = COLAB_RUNTIME_ROOT / "shelf-detection" if IS_COLAB else LOCAL_PROJECT_ROOT
MODEL_NAME = "yolov8m.pt"
DATASET_NAME = "SKU-110K.yaml"
RUNS_DIR = DRIVE_ROOT / "runs"
MODELS_DIR = DRIVE_ROOT / "models"
DRIVE_DATASETS_DIR = DRIVE_ROOT / "datasets"
DRIVE_SKU110K_DIR = DRIVE_DATASETS_DIR / "SKU-110K"
DRIVE_DATASET_YAML = DRIVE_DATASETS_DIR / "SKU-110K.drive.yaml"
LOCAL_SKU110K_DIR = RUNTIME_ROOT / "SKU-110K"
LOCAL_DATASET_YAML = RUNTIME_ROOT / "SKU-110K.local.yaml"

SAVE_SKU110K_TO_DRIVE = True
RESTORE_SKU110K_FROM_DRIVE = True

EPOCHS = 100
BATCH = 8
IMAGE_SIZE = 640
LR0 = 0.01
LRF = 0.01
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3
PATIENCE = 20
DEVICE = "0"
WORKERS = 8
AMP = True
CACHE = False
RUN_NAME = "shelf-detection-colab-v1"
WANDB_PROJECT = "shelfscan"
PYTORCH_INSTALL_ARGS = ["torch", "torchvision", "torchaudio"]
# Untuk Linux + NVIDIA/CUDA, ganti ke wheel resmi PyTorch, misalnya:
# PYTORCH_INSTALL_ARGS = ["torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu128"]
USE_WANDB = False
RESUME_CHECKPOINT = ""

print(f"Runtime      : {'Google Colab' if IS_COLAB else 'local / non-Colab'}")
print(f"Project root : {REPO_DIR}")
print(f"Persistent   : {DRIVE_ROOT}")
print(f"Model        : {MODEL_NAME}")
print(f"Dataset      : {DATASET_NAME}")
print(f"Epoch / batch: {EPOCHS} / {BATCH}")
print(f"Run name     : {RUN_NAME}")
print(f"Dataset cache: {DRIVE_SKU110K_DIR}")

## 3. Siapkan storage persisten

Di Google Colab, cell ini mount Google Drive dan memakai `/content/drive/MyDrive/shelf-detection` sebagai root persisten. Di luar Colab, root persisten otomatis menjadi folder tempat notebook dijalankan.

In [ ]:
if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Mode non-Colab: memakai folder project lokal sebagai storage persisten.")

for directory in (DRIVE_ROOT, RUNS_DIR, MODELS_DIR, DRIVE_ROOT / "wandb", DRIVE_DATASETS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Storage persisten siap: {DRIVE_ROOT}")

## 4. Instal dependency training

Cell ini memasang PyTorch sebelum pemeriksaan GPU supaya notebook lokal tidak gagal pada `import torch`. Default `PYTORCH_INSTALL_ARGS` cocok untuk macOS/CPU. Untuk Linux dengan NVIDIA GPU, ganti nilainya di konfigurasi ke wheel CUDA resmi PyTorch, misalnya `cu128`, sebelum menjalankan cell ini.


In [ ]:
try:
    import torch
except ModuleNotFoundError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PYTORCH_INSTALL_ARGS],
        check=True,
    )
    import torch

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics>=8.4.70",
        "wandb>=0.18.0",
    ],
    check=True,
)

import ultralytics
import wandb
from ultralytics import YOLO

print(f"PyTorch    : {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"W&B        : {wandb.__version__}")


## 5. Periksa GPU

PyTorch harus mendeteksi CUDA. `nvidia-smi` menampilkan tipe GPU, VRAM, driver, dan penggunaan memori. Notebook dihentikan lebih awal jika GPU belum aktif agar training tidak tanpa sengaja berjalan di CPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU CUDA tidak tersedia. Di Colab pilih Runtime > Change runtime type > T4 GPU. "
        "Di lokal, pastikan driver NVIDIA aktif dan PyTorch CUDA wheel sudah terpasang. "
        "Untuk CPU smoke test, gunakan notebook POC atau ubah konfigurasi training secara sadar."
    )

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"CUDA    : {torch.version.cuda}")
subprocess.run(["nvidia-smi"], check=False)


## 6. Siapkan repository

Di Colab, repository terbaru `git@github.com:kholiqdev/shelf-detection.git` di-clone lewat padanan HTTPS publik agar tidak memerlukan SSH key. Di luar Colab, notebook memakai folder kerja saat ini dan hanya membaca commit Git jika tersedia.

In [ ]:
if IS_COLAB:
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    if not (REPO_DIR / ".git").is_dir():
        print(f"Peringatan: {REPO_DIR} bukan repository Git. git_commit akan bernilai unknown.")

os.chdir(REPO_DIR)
git_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=False,
)
git_commit = git_result.stdout.strip() if git_result.returncode == 0 else "unknown"
print(f"Repository : {REPO_DIR}")
print(f"Git commit : {git_commit}")

## 7. Verifikasi dependency khusus training

Cell ini hanya memverifikasi versi dependency yang sudah dipasang pada langkah sebelumnya. Dependency CLIP, FAISS, FastAPI, dan ONNX tidak diperlukan untuk training detector sehingga sengaja tidak dipasang.

In [ ]:
print(f"PyTorch    : {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"W&B        : {wandb.__version__}")


## 8. Atur W&B

W&B dipakai dengan project `shelfscan` agar struktur experiment tracking sama dengan referensi. Secara default mode offline dan log-nya disimpan di Drive. Untuk dashboard online, buat secret `WANDB_API_KEY` melalui ikon kunci di sidebar Colab lalu ubah `USE_WANDB = True`. API key tidak ditulis langsung di notebook.

In [ ]:
os.environ["WANDB_DIR"] = str(DRIVE_ROOT / "wandb")
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb_config = {
    "model": MODEL_NAME,
    "data": DATASET_NAME,
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH,
    "lr0": LR0,
    "lrf": LRF,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "patience": PATIENCE,
    "git_commit": git_commit,
}

if USE_WANDB:
    if IS_COLAB:
        from google.colab import userdata

        api_key = userdata.get("WANDB_API_KEY")
    else:
        api_key = os.environ.get("WANDB_API_KEY")
    if not api_key:
        raise RuntimeError("WANDB_API_KEY belum tersedia di Colab Secrets atau environment variable.")
    os.environ["WANDB_API_KEY"] = api_key
    os.environ["WANDB_MODE"] = "online"
else:
    os.environ["WANDB_MODE"] = "offline"

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config=wandb_config,
    mode=os.environ["WANDB_MODE"],
)
print(f"W&B {os.environ['WANDB_MODE']}. Log disimpan di {os.environ['WANDB_DIR']}")

## 9. Unduh, restore, dan backup SKU-110K resmi

Ultralytics menyediakan `SKU-110K.yaml` dengan script download dan konversi anotasi resmi. Cell ini memakai cache storage persisten bila tersedia, atau mengunduh sekitar 13.6 GB ke storage runtime/lokal pada penggunaan pertama. Setelah dataset siap, cell dapat menyalinnya ke storage persisten sebagai backup untuk runtime berikutnya.

In [ ]:
from ultralytics.data.utils import check_det_dataset

EXPECTED_SPLITS = ("train", "val", "test")


def is_sku110k_prepared(path: Path) -> bool:
    return path.is_dir() and all((path / f"{split}.txt").is_file() for split in EXPECTED_SPLITS)


def write_sku110k_yaml(yaml_path: Path, dataset_path: Path) -> Path:
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text(
        f"path: {dataset_path}\n"
        "train: train.txt\n"
        "val: val.txt\n"
        "test: test.txt\n"
        "names:\n"
        "  0: object\n"
    )
    return yaml_path


def rsync_directory(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["rsync", "-ah", "--info=progress2", f"{source}/", f"{destination}/"],
        check=True,
    )


DATASET_FOR_TRAINING = DATASET_NAME
if RESTORE_SKU110K_FROM_DRIVE and is_sku110k_prepared(DRIVE_SKU110K_DIR):
    if not is_sku110k_prepared(LOCAL_SKU110K_DIR):
        print(f"Menyalin SKU-110K dari Drive ke lokal: {LOCAL_SKU110K_DIR}")
        rsync_directory(DRIVE_SKU110K_DIR, LOCAL_SKU110K_DIR)
    DATASET_FOR_TRAINING = str(write_sku110k_yaml(LOCAL_DATASET_YAML, LOCAL_SKU110K_DIR))
else:
    print("Cache SKU-110K di Drive belum tersedia. Ultralytics akan menyiapkan dataset dari descriptor resmi.")

dataset_info = check_det_dataset(DATASET_FOR_TRAINING)
dataset_root = Path(dataset_info["path"])
dataset_counts = {}

for split in EXPECTED_SPLITS:
    split_value = dataset_info.get(split)
    if not split_value:
        raise RuntimeError(f"Split {split} tidak tersedia pada {DATASET_FOR_TRAINING}.")

    split_sources = split_value if isinstance(split_value, list) else [split_value]
    image_count = 0
    for split_source in split_sources:
        split_path = Path(split_source)
        if not split_path.is_absolute():
            split_path = dataset_root / split_path
        if split_path.is_file():
            image_count += sum(
                1
                for line in split_path.read_text().splitlines()
                if line.strip()
            )
        elif split_path.is_dir():
            image_count += sum(
                1
                for path in split_path.rglob("*")
                if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            )
        else:
            raise FileNotFoundError(f"Sumber split {split} tidak ditemukan: {split_path}")

    if image_count == 0:
        raise RuntimeError(f"Split {split} kosong.")
    dataset_counts[split] = {"images": image_count}
    print(f"{split:>5}: {image_count:>6} images")

if SAVE_SKU110K_TO_DRIVE and not is_sku110k_prepared(DRIVE_SKU110K_DIR):
    print(f"Menyimpan SKU-110K ke storage persisten: {DRIVE_SKU110K_DIR}")
    rsync_directory(dataset_root, DRIVE_SKU110K_DIR)
else:
    print("Backup SKU-110K ke storage persisten dilewati atau sudah tersedia.")

write_sku110k_yaml(DRIVE_DATASET_YAML, DRIVE_SKU110K_DIR)
print(f"Dataset root aktif : {dataset_root}")
print(f"Dataset training   : {DATASET_FOR_TRAINING}")
print(f"YAML persisten     : {DRIVE_DATASET_YAML}")

## 10. Buat konfigurasi training

Konfigurasi training dibuat langsung di notebook agar tidak bergantung pada file lain di repository. Nilainya mengikuti konfigurasi referensi: YOLOv8m, 100 epoch, batch 8, learning-rate schedule, momentum, weight decay, warmup, patience, AMP aktif, dan cache nonaktif. Output `project` diarahkan ke storage persisten, sedangkan dataset aktif berada di storage runtime/lokal.

In [ ]:
training_config = {
    "epochs": EPOCHS,
    "batch": BATCH,
    "imgsz": IMAGE_SIZE,
    "lr0": LR0,
    "lrf": LRF,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "patience": PATIENCE,
    "device": DEVICE,
    "workers": WORKERS,
    "amp": AMP,
    "cache": CACHE,
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": False,
    "verbose": True,
}
print(json.dumps({"model": MODEL_NAME, "data": DATASET_FOR_TRAINING, **training_config}, indent=2))

## 11. Jalankan training

Untuk training baru, cell memanggil API Ultralytics langsung dengan konfigurasi yang sama di Colab maupun lokal dan descriptor resmi SKU-110K. Ultralytics akan mengunduh pretrained `yolov8m.pt` pada run pertama.

Jika `RESUME_CHECKPOINT` diisi dengan path `last.pt`, Ultralytics melanjutkan epoch dan state optimizer dari checkpoint tersebut. Jangan mengisi `RESUME_CHECKPOINT` untuk run baru.

In [ ]:
if RESUME_CHECKPOINT:
    checkpoint_path = Path(RESUME_CHECKPOINT)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {checkpoint_path}")
    print(f"Melanjutkan training dari {checkpoint_path}")
    train_results = YOLO(str(checkpoint_path)).train(resume=True)
else:
    print("Memulai training baru dengan Ultralytics.")
    train_results = YOLO(MODEL_NAME).train(data=DATASET_FOR_TRAINING, **training_config)

training_metrics = {
    "mAP50": float(train_results.results_dict.get("metrics/mAP50(B)", 0)),
    "mAP50-95": float(train_results.results_dict.get("metrics/mAP50-95(B)", 0)),
    "precision": float(train_results.results_dict.get("metrics/precision(B)", 0)),
    "recall": float(train_results.results_dict.get("metrics/recall(B)", 0)),
}
wandb.log(training_metrics)
print(json.dumps(training_metrics, indent=2))

## 12. Simpan model dan metadata

Notebook mencari `best.pt` terbaru di folder run, lalu menyalinnya ke dua lokasi:

- `models/<RUN_NAME>/best.pt` untuk menyimpan versi run.
- `models/best.pt` sebagai path sederhana untuk notebook POC.

SHA-256, commit Git, parameter training, statistik dataset, dan metrik akhir disimpan sebagai `metadata.json`. Checksum membantu memastikan file model tidak berubah atau rusak saat dipindahkan.

In [ ]:
best_candidates = list(RUNS_DIR.glob("*/weights/best.pt"))
if not best_candidates:
    raise FileNotFoundError(f"best.pt tidak ditemukan di {RUNS_DIR}")

best_source = max(best_candidates, key=lambda path: path.stat().st_mtime)
version_dir = MODELS_DIR / RUN_NAME
version_dir.mkdir(parents=True, exist_ok=True)
versioned_model = version_dir / "best.pt"
latest_model = MODELS_DIR / "best.pt"
shutil.copy2(best_source, versioned_model)
shutil.copy2(best_source, latest_model)
sha256 = hashlib.sha256(versioned_model.read_bytes()).hexdigest()
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_name": RUN_NAME,
    "git_commit": git_commit,
    "model_name": MODEL_NAME,
    "model_source": str(best_source),
    "model_path": str(versioned_model),
    "sha256": sha256,
    "training_config": {"model": MODEL_NAME, "data": DATASET_FOR_TRAINING, **training_config},
    "dataset_name": DATASET_NAME,
    "dataset_for_training": DATASET_FOR_TRAINING,
    "dataset_root": str(dataset_root),
    "drive_dataset_root": str(DRIVE_SKU110K_DIR),
    "drive_dataset_yaml": str(DRIVE_DATASET_YAML),
    "training_metrics": training_metrics,
    "dataset_counts": dataset_counts,
}
metadata_path = version_dir / "metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))
wandb_run.summary.update(training_metrics)
wandb.finish()

print(f"Model versi : {versioned_model}")
print(f"Model latest: {latest_model}")
print(f"Metadata    : {metadata_path}")
print(f"SHA-256     : {sha256}")

## Selesai

Training selesai ketika cell sebelumnya menampilkan path model dan SHA-256. File yang akan dipakai oleh notebook POC adalah:

```text
Google Colab : /content/drive/MyDrive/shelf-detection/models/best.pt
Lokal        : <folder-project>/models/best.pt
```

Untuk smoke test, ubah `EPOCHS` menjadi 5 dan gunakan nama run terpisah. Untuk hasil final yang sejalan dengan konfigurasi referensi, kembalikan `EPOCHS` ke 100.